# Demonstração: Pipeline de Dados Ariranha DS

Este notebook demonstra passo a passo como os dados de focos de incêndio (INPE) são processados, limpos, agrupados espacialmente (clusterização via grid) e utilizados para treinar os modelos de Risco de Ocorrência e Severidade (FRP).

## 1. Importações e Configurações

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adicionando a raiz do projeto ao path para importar modulos do src
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.config import DATA_DIR
from src.data_loader import load
from src.features import add_grid_cell, add_temporal_features

print("Configurações carregadas com sucesso.")

## 2. Leitura e Limpeza dos Dados
A função `load` acessa o `.parquet`, remove linhas com coordenadas ausentes, deduplica os focos no mesmo momento e local e preenche valores faltantes nas features críticas.

In [ ]:
# Carregando os dados brutos (pode demorar um pouco dependendo do tamanho do dataset)
df_bruto = load()

print(f"Total de registros após limpeza inicial: {len(df_bruto):,}")
display(df_bruto.head())

## 3. Clusterização Espacial (Grid) e Feature Engineering
Para treinar nosso modelo de predição espacial, agrupamos os focos de incêndio em células de 0.25° (aproximadamente 27km).


In [ ]:
# Aplicando Grid (Clusterização Espacial)
df_features = add_grid_cell(df_bruto)

# Extração de features temporais
df_features = add_temporal_features(df_features)

n_celulas = df_features['cell_id'].nunique()
print(f"Focos agrupados em {n_celulas:,} clusters/células espaciais únicas.")
display(df_features[['data_hora', 'latitude', 'longitude', 'lat_grid', 'lon_grid', 'cell_id', 'mes', 'frp']].head())

### Visualização dos Clusters de Focos (Mapa Simplificado)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(df_features['lon_grid'], df_features['lat_grid'], alpha=0.1, s=1, c='red')
plt.title('Distribuição Espacial dos Clusters (Grid 0.25°)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 4. Agregação e Construção do Perfil Histórico (Cell Stats)
A partir das células clusterizadas, construímos um perfil histórico de cada local.

In [ ]:
from src.train import computar_cell_stats

cell_stats = computar_cell_stats(df_features)
display(cell_stats.head())

## 5. Treinamento dos Modelos (Ocorrência e Severidade)
1. **Construímos os exemplos**: Amostramos dias sem foco para a base negativa.
2. **Treino de Ocorrência**: Classificador que prevê a probabilidade de haver foco nas próximas 24h.
3. **Treino de Severidade (FRP)**: Regressor que estima a potência do fogo (FRP) em MW.

In [ ]:
from src.train import construir_exemplos, treinar, treinar_severidade
from sklearn.preprocessing import LabelEncoder

rng = np.random.default_rng(42)
exemplos = construir_exemplos(df_features, rng)

encoder = LabelEncoder()
encoder.fit(cell_stats["bioma_modal"])

print("\n--- Iniciando treinamento simplificado ---\n")
modelo_occ, metricas_occ = treinar(exemplos, cell_stats, encoder)
modelo_sev, metricas_sev = treinar_severidade(exemplos, cell_stats, encoder)

## 6. O que os modelos retornam?

Ao realizar requisições na API para uma determinada coordenada, os dois modelos são executados.

### 6.1 Modelo de Ocorrência (XGBClassifier)
**Retorno:** Probabilidade (0 a 1) e uma Classe Categórica.
- Estima o quão provável é a ocorrência de focos na célula nas próximas 24h.
- **Risco Baixo:** prob < 0.30
- **Risco Moderado:** 0.30 <= prob < 0.60
- **Risco Alto:** 0.60 <= prob < 0.80
- **Risco Crítico:** prob >= 0.80

### 6.2 Modelo de Severidade (XGBRegressor)
**Retorno:** Valor Contínuo (Megawatts) e Classe de Severidade.
- Entra em ação caso exista a possibilidade do foco (prob >= 0.30).
- Retorna a Potência Radiativa do Fogo (FRP - Fire Radiative Power).
- **Severidade Baixa:** FRP < 30 MW
- **Severidade Moderada:** 30 <= FRP < 100 MW
- **Severidade Alta:** 100 <= FRP < 500 MW
- **Severidade Extrema:** FRP >= 500 MW

## 7. Gráficos de Predições (Ocorrência e Severidade)
Vamos gerar previsões na base simulada e visualizar os resultados.

In [ ]:
# Vamos selecionar uma amostra de 50 mil casos do dataset para ver os resultados previstos
amostra_teste = exemplos.sample(50000, random_state=42).copy()
cell_stats_tmp = cell_stats.copy()
cell_stats_tmp["bioma_enc"] = encoder.transform(cell_stats_tmp["bioma_modal"])

from src.config import FEATURES
feature_cols_celula = ["cell_id", "media_risco", "media_dias_seco", "media_precip", "media_frp_historico", "n_historico", "bioma_enc"]
df_test = amostra_teste.merge(cell_stats_tmp[feature_cols_celula], on="cell_id", how="left")
df_test["mes"] = df_test["data"].dt.month
df_test["dia_do_ano"] = df_test["data"].dt.dayofyear

X_test = df_test[FEATURES].fillna(-1)

# Predições do modelo de Ocorrência (Probabilidade 0-1)
probabilidades = modelo_occ.predict_proba(X_test)[:, 1]
df_test["prob_ocorrencia"] = probabilidades

# Predições do modelo de Severidade (Somente faz sentido onde prevemos focos, mas vamos prever todos para análise)
severidades_pred = modelo_sev.predict(X_test)
# Evitar FRP negativo
severidades_pred = np.maximum(0, severidades_pred)
df_test["frp_previsto"] = severidades_pred

In [ ]:
# 7.1 Distribuição da Probabilidade de Ocorrência
plt.figure(figsize=(12, 5))
sns.histplot(data=df_test, x="prob_ocorrencia", hue="incendio", bins=50, kde=True, palette="viridis")
plt.title('Distribuição da Probabilidade Prevista vs Fogo Real (0 = Sem Fogo, 1 = Fogo)')
plt.xlabel('Probabilidade Prevista de Ocorrência')
plt.ylabel('Frequência')
plt.axvline(x=0.30, color='r', linestyle='--', label='Limiar Moderado (0.3)')
plt.axvline(x=0.80, color='darkred', linestyle='--', label='Limiar Crítico (0.8)')
plt.legend()
plt.show()

In [ ]:
# 7.2 Distribuição de Severidade (FRP) para casos Positivos reais
positivos = df_test[df_test['incendio'] == 1].copy()

plt.figure(figsize=(10, 6))
plt.scatter(positivos['frp_target'], positivos['frp_previsto'], alpha=0.3, color='orange')
plt.plot([0, 2000], [0, 2000], 'k--', alpha=0.5) # Linha de identidade
plt.title('FRP Real vs FRP Previsto (Megawatts)')
plt.xlabel('Severidade Real (FRP Observado)')
plt.ylabel('Severidade Prevista (FRP)')
plt.xlim(0, max(positivos['frp_previsto'].max(), 500))
plt.ylim(0, max(positivos['frp_previsto'].max(), 500))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 7.3 Mapa de Calor das Classes de Risco vs Severidade Prevista
def cat_risco(p):
    if p < 0.3: return 'Baixo'
    elif p < 0.6: return 'Moderado'
    elif p < 0.8: return 'Alto'
    else: return 'Critico'

def cat_sev(s):
    if s < 30: return 'Baixa'
    elif s < 100: return 'Moderada'
    elif s < 500: return 'Alta'
    else: return 'Extrema'

df_test['Risco'] = df_test['prob_ocorrencia'].apply(cat_risco)
df_test['Severidade_Classe'] = df_test['frp_previsto'].apply(cat_sev)

crosstab = pd.crosstab(df_test['Risco'], df_test['Severidade_Classe'])
# Ordenando eixos
crosstab = crosstab.reindex(['Baixo', 'Moderado', 'Alto', 'Critico'])
if 'Extrema' not in crosstab.columns: crosstab['Extrema'] = 0
crosstab = crosstab[['Baixa', 'Moderada', 'Alta', 'Extrema']]

plt.figure(figsize=(8, 6))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Matriz: Risco de Ocorrência vs Severidade Prevista (Amostra)')
plt.show()